# Notebook 11 — Vedere lo shift e adattarsi (digits)

Il Notebook 5 si fermava a *misurare* lo shift (AUROC, rejection curve,
rotazione di MNIST) senza mai adattare il modello. Qui si fa un passo in
più: dopo aver confermato lo shift su MNIST/USPS (stessa idea del Notebook
5, con `src/metrics.py`), si adatta davvero il modello con
`adapt_target`/`im_loss` (`src/digits_adapt.py`, portati da
`src/im_adapt.py` del progetto principale -- `code_v2` non aveva un
adattamento vero e proprio, solo la misura dello shift).

Convenzione: `g` sbloccato, `h` congelato (gestito da `adapt_target`
stesso), `weight_mode="uncertainty"` (default della funzione, U-SFAN),
`gamma=0.5`, `temperature=0.4`, `lr=1e-2`, `M=100`, `steps=300` -- stessa
convenzione già usata per HAR nel progetto principale. **Prima di lanciare
i 300 step per intero**, misura il tempo di un singolo step full-batch
sulla dimensione reale del dominio target (`steps=1` più sotto): se il
costo non è ragionevole su questa architettura (più piccola di quella usata
in una versione precedente di questo stesso esperimento), valuta un
sottocampionamento invece di assumere che vada bene.

## Setup: ricarica checkpoint, fit di Laplace

In [1]:
import sys, copy, time
from pathlib import Path

cwd = Path().resolve()
PROJ = cwd
while not (PROJ / "src").exists():
    PROJ = PROJ.parent
sys.path.insert(0, str(PROJ))

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, TensorDataset

from src.digits_data import load_domain
from src.digits_model import SmallCNN32
from src.bayesian_model import extract, augment, LastLayerLaplace
from src.digits_adapt import adapt_target
from src.metrics import auroc, rejection_curve

MODELS_DIR = PROJ / "models" / "source_svhn"
ckpt = torch.load(MODELS_DIR / "model.pt", map_location="cpu", weights_only=False)
lap_data = np.load(MODELS_DIR / "svhn_laplace.npz")
mc_conv = np.load(MODELS_DIR / "mc_convergence.npz")
mean, std = ckpt["source_mean"], ckpt["source_std"]
M_FIXED = int(mc_conv["M_FIXED"])

model = SmallCNN32(n_classes=ckpt["n_classes"], feature_dim=ckpt["feature_dim"])
model.load_state_dict(ckpt["state_dict"])
model.eval()
laplace = LastLayerLaplace(theta_map=lap_data["theta_map"], cov=lap_data["cov"],
                           K=int(lap_data["K"]), Dp=int(lap_data["Dp"]))

TARGET_DOMAINS = ["mnist", "usps"]
target_raw = {d: load_domain(d, "test", mean, std) for d in TARGET_DOMAINS}
print(f"M_FIXED (dal Notebook 9) = {M_FIXED}")

M_FIXED (dal Notebook 9) = 1000


## 1. Vedere lo shift (prima dell'adattamento)

Stessa idea del Notebook 5: usa l'incertezza epistemica come score per
rilevare i punti fuori dal manifold source (AUROC, con "positivo" = target)
e la rejection curve (accuratezza in funzione della copertura, scartando i
punti più incerti).

In [2]:
def predict_domain(X, y):
    loader = DataLoader(TensorDataset(X, y), batch_size=256)
    Phi, y_np, _ = extract(model, loader, device="cpu")
    Phi_aug = augment(Phi)
    rng = np.random.default_rng(456)
    pred = laplace.predictive_batched(Phi_aug, M=M_FIXED, rng=rng)
    return pred, y_np


X_svhn_test, y_svhn_test = load_domain("svhn", "test", mean, std)
pred_svhn, y_svhn_np = predict_domain(X_svhn_test, y_svhn_test)

pre_results = {}
for domain in TARGET_DOMAINS:
    X_t, y_t = target_raw[domain]
    pred_t, y_t_np = predict_domain(X_t, y_t)
    pre_results[domain] = dict(pred=pred_t, y=y_t_np)

    scores = np.concatenate([pred_svhn["epistemic"], pred_t["epistemic"]])
    labels = np.concatenate([np.zeros(len(y_svhn_np)), np.ones(len(y_t_np))])
    a = auroc(scores, labels)
    acc_pre = (pred_t["probs"].argmax(axis=1) == y_t_np).mean()
    print(f"{domain}: AUROC(epistemica, svhn vs {domain})={a:.3f}  accuracy pre-adattamento={100*acc_pre:.2f}%")

mnist: AUROC(epistemica, svhn vs mnist)=0.580  accuracy pre-adattamento=58.63%
usps: AUROC(epistemica, svhn vs usps)=0.648  accuracy pre-adattamento=59.64%


## 2. Adattamento: `adapt_target` su MNIST test e USPS test -- SHOT-IM vs. U-SFAN vs. epistemic-only

Tre bracci per ciascun dominio target, tutti da una copia fresca del
source model, stesso seed e stessi altri iperparametri, unica differenza
`weight_mode`:

- **SHOT-IM** (`weight_mode="none"`): IM non pesato, il baseline
  convenzionale.
- **U-SFAN** (`weight_mode="uncertainty"`): IM pesato dall'incertezza
  epistemica del source (Eq. 6-7 del paper) -- quello già presente prima.
- **epistemic-only** (`weight_mode="epistemic_only"`): terzo braccio,
  ablation diagnostica -- vedi cella markdown sotto per la motivazione
  completa.

Senza SHOT-IM come confronto diretto non è possibile sostenere che la
pesatura guidata dall'incertezza aiuti *rispetto all'IM semplice* -- solo
che adattare in generale aiuta rispetto a non fare nulla.

### Il terzo braccio (`epistemic_only`) è una deviazione dal paper, non una replica alternativa

U-SFAN (Eq. 6-7) pesa il termine di entropia con `w_i = exp(-H_i)`, dove
`H_i` è l'entropia predittiva **totale** (epistemica + aleatoria). Questo
terzo braccio usa invece `w_i = exp(-z_i)`, con `z_i` la sola componente
**epistemica**, standardizzata (z-score) contro un riferimento **fisso**
calcolato una volta sola sulla distribuzione di epistemica del source
(`pred_svhn["epistemic"]`, già disponibile dalla Sezione 1 -- mediana e
IQR, mai ricalcolati per-batch/per-step sul target, altrimenti si
misurerebbe "quanto un punto target è insolito rispetto ad altri punti
target", non rispetto al source). L'epistemica grezza non viene mai usata
direttamente: non è su una scala comparabile fra punti/domini (stessa
ragione di `normalize_epistemic` in `src/bayesian.py` del progetto
principale).

Non è una replica alternativa del metodo -- è un'**ablation diagnostica**
per testare l'ipotesi che la dominanza della componente aleatoria
nell'entropia totale (vedi discussione sulla Sezione 3, "Calibrazione dopo
l'adattamento") sia la causa per cui la pesatura di U-SFAN fa poco o
peggio di SHOT-IM: se isolare la sola epistemica cambia il risultato in
modo apprezzabile rispetto a U-SFAN, è un indizio che il segnale utile
c'è ma viene diluito dall'aleatoria nell'entropia totale; se non cambia
nulla, il problema è altrove.

In [ ]:
ADAPT_STEPS = 50
BASE_KWARGS = dict(gamma=0.5, temperature=0.4, lr=1e-2, M=100)
ARM_WEIGHT_MODE = {"shot_im": "none", "u_sfan": "uncertainty", "epistemic_only": "epistemic_only"}

# Riferimento fisso per lo z-score del braccio epistemic_only: mediana/IQR
# dell'epistemica sul SOURCE (svhn test, pred_svhn dalla Sezione 1),
# calcolato una volta sola qui -- mai per-batch/per-step sul target.
source_epi_median = float(np.median(pred_svhn["epistemic"]))
source_epi_iqr = float(np.percentile(pred_svhn["epistemic"], 75) - np.percentile(pred_svhn["epistemic"], 25))
print(f"riferimento fisso (source svhn): mediana epistemica={source_epi_median:.4f}  "
      f"IQR={source_epi_iqr:.4f}")
print(f"{len(ARM_WEIGHT_MODE)} bracci x {len(TARGET_DOMAINS)} domini x {ADAPT_STEPS} step ciascuno\n")

adaptation_results = {}
for domain, seed in [("mnist", 1), ("usps", 2)]:
    adaptation_results[domain] = {}
    for arm, weight_mode in ARM_WEIGHT_MODE.items():
        print(f"\n{'=' * 60}\nAdattamento su {domain}, braccio {arm} (weight_mode={weight_mode!r}, "
              f"full batch, steps={ADAPT_STEPS}, seed={seed})\n{'=' * 60}")
        X_t, y_t = target_raw[domain]
        m = copy.deepcopy(model)
        extra_kwargs = dict(source_epi_median=source_epi_median, source_epi_iqr=source_epi_iqr) \
            if weight_mode == "epistemic_only" else {}
        hist = adapt_target(m, laplace, X_t, weight_mode=weight_mode, steps=ADAPT_STEPS,
                            seed=seed, **BASE_KWARGS, **extra_kwargs)

        m.eval()
        with torch.no_grad():
            probs_post = torch.softmax(m(X_t), dim=-1).numpy()
        with torch.no_grad():
            phi_post = m.features(X_t).numpy()
        Phi_aug_post = augment(phi_post)

        acc_pre = (pre_results[domain]["pred"]["probs"].argmax(axis=1) == pre_results[domain]["y"]).mean()
        acc_post = (probs_post.argmax(axis=1) == y_t.numpy()).mean()

        adaptation_results[domain][arm] = dict(hist=hist, probs_post=probs_post, Phi_aug_post=Phi_aug_post,
                                               y=y_t.numpy(), acc_pre=acc_pre, acc_post=acc_post)
        print(f"{domain}/{arm}: pre={100 * acc_pre:.2f}%  post={100 * acc_post:.2f}%  "
              f"delta={100 * (acc_post - acc_pre):+.2f}pp")

## Traiettoria di loss/entropia/diversità

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fields = ["loss", "ent", "div", "mean_weight"]
titles = ["loss IM", "termine entropia (pesato)", "termine diversità", "peso medio per campione"]
colors = {"mnist": "tab:purple", "usps": "tab:red"}
linestyles = {"shot_im": "--", "u_sfan": "-", "epistemic_only": ":"}

for ax, field, title in zip(axes, fields, titles):
    for domain in TARGET_DOMAINS:
        for arm in ARM_WEIGHT_MODE:
            if field == "mean_weight" and arm == "shot_im":
                continue  # shot_im non pesa mai i campioni: costantemente 1.0, non informativo
            ax.plot(adaptation_results[domain][arm]["hist"][field], color=colors[domain],
                    linestyle=linestyles[arm], label=f"{domain} ({arm})")
    ax.set_xlabel("step di adattamento")
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=6)
fig.suptitle(f"Traiettorie di adattamento (steps={ADAPT_STEPS}, full batch) -- "
            f"tratteggiato = SHOT-IM, continuo = U-SFAN, punteggiato = epistemic-only")
fig.tight_layout()
plt.show()

## 3. Calibrazione dopo l'adattamento

Questa sezione riporta **solo accuracy ed ECE** post-adattamento. Il
confronto "epistemica pre vs. post adattamento" (calcolato con la STESSA
posterior di Laplace fittata una volta sola sul source) è stato **rimosso
dalle metriche principali, non solo rivisto** -- motivazione qui sotto.

### Perché l'epistemica pre/post non è una metrica valida qui

L'incertezza epistemica, in questa pipeline, gioca un ruolo puramente
strumentale: entra nel calcolo del peso per-campione durante
`adapt_target` (Eq. 6-7 di U-SFAN), ricalcolata a ogni step sulle feature
CORRENTI del modello che si sta adattando. Una volta che l'adattamento è
concluso, chiedersi "quanto è incerto il modello adesso" usando la STESSA
posterior fissata sul source (mai riaggiornata) non è una domanda ben
posta: quella posterior descrive la curvatura della log-posterior attorno
a feature che il `g` adattato non produce più. Il numero che ne risultava
(su questa run: epistemica +0.23 nats su entrambi i domini, un salto
enorme rispetto ai valori pre-adattamento) riflette quanto le nuove
feature sono estrapolate rispetto alla regione coperta dal fit originale,
non un giudizio bayesiano coerente sull'affidabilità del modello adattato.

Non è un'osservazione ad hoc -- due lavori NeurIPS 2021 la supportano
direttamente:

- Izmailov, Nicholson, Lotfi, Wilson, *"Dangers of Bayesian Model
  Averaging under Covariate Shift"* (NeurIPS 2021, arXiv:2106.11905):
  mostrano che una posterior bayesiana ad alta fedeltà (HMC) può
  comportarsi in modo patologico sotto covariate shift, arrivando a
  sottoperformare drasticamente una semplice soluzione MAP (su CIFAR-10-C
  alla severità massima, -25 punti di accuracy) -- una posterior fittata
  su una distribuzione non ha comportamento garantito su feature fuori da
  quella distribuzione.
- Zhou & Levine, *"Training on Test Data with Bayesian Adaptation for
  Covariate Shift"* (BACS, NeurIPS 2021, arXiv:2109.12746): notano
  esplicitamente che adattare minimizzando l'entropia SENZA aggiornare il
  termine di posterior può portare a soluzioni degeneri, e che un
  trattamento bayesiano coerente dell'adattamento richiederebbe di
  aggiornare la posterior stessa usando i dati target -- cosa che né BACS
  in quella forma semplice né U-SFAN (né questa pipeline) fanno.

Accuracy ed ECE, invece, restano confronti pre/post validi: sono definiti
sul comportamento osservato del modello (previsioni corrette, probabilità
calibrate), non su un oggetto bayesiano interno che perde validità quando
il feature extractor cambia.

In [ ]:
from src.metrics import expected_calibration_error

# ECE resta calcolato sul braccio U-SFAN (quello già presente prima
# dell'aggiunta del confronto a due bracci) -- non richiesto un confronto
# a tre vie su ECE in questo passo, solo sull'accuracy (tabella sotto).
post_results = {}
for domain in TARGET_DOMAINS:
    r = adaptation_results[domain]["u_sfan"]
    ece_post = expected_calibration_error(r["probs_post"], r["y"])
    post_results[domain] = dict(ece=ece_post)

    ece_pre = expected_calibration_error(pre_results[domain]["pred"]["probs"], pre_results[domain]["y"])
    print(f"{domain}: ECE pre={ece_pre:.4f} post (U-SFAN)={ece_post:.4f} delta={ece_post-ece_pre:+.4f}")

## Riepilogo: source-only vs. SHOT-IM vs. U-SFAN vs. epistemic-only

In [ ]:
print(f"{'dominio':>10s} {'source-only':>13s} {'post SHOT-IM':>14s} {'post U-SFAN':>14s} "
      f"{'post epistemic-only':>21s}")
print("-" * 78)
for domain in TARGET_DOMAINS:
    pre = adaptation_results[domain]["u_sfan"]["acc_pre"]  # stesso source per tutti i bracci
    post_shot = adaptation_results[domain]["shot_im"]["acc_post"]
    post_usfan = adaptation_results[domain]["u_sfan"]["acc_post"]
    post_epi = adaptation_results[domain]["epistemic_only"]["acc_post"]
    print(f"{domain:>10s} {100 * pre:12.2f}% {100 * post_shot:13.2f}% {100 * post_usfan:13.2f}% "
          f"{100 * post_epi:20.2f}%")

### Discussione: perché il vantaggio di U-SFAN dipende dal regime di incertezza del source

Il confronto sopra mostra SHOT-IM (peso uniforme) ed epistemic_only (peso sulla sola
componente epistemica) battere sistematicamente U-SFAN (peso sull'entropia totale) su
entrambi i target. Questo risultato, apparentemente in contraddizione con l'assunto
centrale di U-SFAN, è coerente con un meccanismo documentato in letteratura.

**Perché SVHN produce un'incertezza dominata dalla componente aleatoria.** Kendall & Gal
(2017, "What Uncertainties Do We Need in Bayesian Deep Learning for Computer Vision?",
NeurIPS) mostrano che l'incertezza epistemica di un modello si riduce sistematicamente
all'aumentare della dimensione del training set, mentre l'aleatoria resta pressoché
costante (la loro Tabella 3, confrontando modelli allenati su frazioni progressive dello
stesso dataset) -- concludendo che "in molti regimi con grandi quantità di dati... è
l'incertezza aleatoria a dominare, essendo l'epistemica in gran parte spiegata via dai dati
disponibili". SVHN, con 65.932 immagini nel nostro split di training, è precisamente questo
regime: il rapporto aleatoria/epistemica osservato sul source (~64x, Notebook 9) è coerente
con un'epistemica "consumata" dalla numerosità dei dati, mentre l'aleatoria riflette una
proprietà stabile del dominio (foto reali di insegne stradali, spesso tagliate ai bordi o
mal illuminate) che nessuna quantità di training può ridurre.

**Perché questo penalizza il peso a entropia totale di U-SFAN.** Il peso w_i = exp(-H_i)
(Eq. 6-7, Roy et al., U-SFAN) usa l'entropia predittiva TOTALE, che su questo source è
dominata dall'aleatoria per il motivo sopra. Questo significa che il down-weighting
penalizza sistematicamente i campioni intrinsecamente ambigui (es. cifre confondibili come
4/9), non necessariamente quelli fuori dal manifold source -- sopprimendo segnale di
apprendimento utile insieme al rumore.

**Un precedente diretto in un contesto diverso.** Guo et al. (2023, "Explore Epistemic
Uncertainty in Domain Adaptive Semantic Segmentation", CIKM) riportano un confronto quasi
analogo in un compito diverso (segmentazione invece di classificazione): pesare il
self-training per Maximum Class Probability (un proxy dell'aleatoria) dà 52.2 mIoU, isolare
l'epistemica dà 54.6 mIoU (loro Tabella 5) -- motivando la scelta con lo stesso argomento
che useremmo qui: l'aleatoria "può portare a overconfidence su previsioni scorrette",
mentre l'epistemica "riflette accuratamente il gap di dominio". Il nostro braccio
epistemic_only, che batte sia shot_im sia u_sfan su mnist (+20.67pp), è una seconda
controprova indipendente dello stesso principio, su un'architettura e un compito diversi da
quelli di Guo et al.

**Il limite non coperto da questi riferimenti: il collasso su USPS.** Nessuno dei due
lavori citati discute un caso in cui isolare l'epistemica PEGGIORA l'adattamento rispetto a
non pesare affatto, come osservato qui su USPS (epistemic_only: -7.27pp, l'unico braccio
sotto la baseline source-only). L'ipotesi più plausibile, per analogia, viene da Franchi et
al. (2022, "Latent Discriminant Deterministic Uncertainty", ECCV): notano che con dati di
training limitati la stima dell'incertezza può risultare instabile o distorta, al punto da
richiedere nel loro caso un secondo stadio di training su outlier sintetici per stabilizzarla.
USPS (2.007 immagini, un quinto di MNIST) è il target più piccolo testato; è plausibile che
la standardizzazione dell'epistemica (mediana/IQR fissate sul source) produca qui una stima
del peso più rumorosa, amplificando l'instabilità del gradiente durante l'adattamento.
Questa è un'estensione per analogia del meccanismo di Franchi et al. (che riguarda
l'instabilità della stima, non il suo uso come peso in un adattamento successivo), non una
dimostrazione diretta.

**Riferimenti:**
- Kendall, A., & Gal, Y. (2017). What Uncertainties Do We Need in Bayesian Deep Learning
  for Computer Vision? NeurIPS.
- Guo, K., Su, Z., Yang, X., Sun, J., & Huang, K. (2023). Explore Epistemic Uncertainty in
  Domain Adaptive Semantic Segmentation. CIKM.
- Franchi, G., Yu, X., Bursuc, A., Aldea, E., Dubuisson, S., & Filliat, D. (2022). Latent
  Discriminant Deterministic Uncertainty. ECCV.

*(placeholder -- da riscrivere con il confronto reale dopo l'esecuzione della cella sopra)*

## TODO prima che questo sia un risultato di tesi

Questo intero notebook usa **un solo seed di training del source (2019) e
una sola run per condizione** -- un solo fit di Laplace, una sola run di
adattamento per dominio target, nessuna ripetizione. Ogni numero e ogni
delta pre/post sopra (accuracy, epistemica, ECE) è quindi una singola
stima puntuale, non una distribuzione: da questo notebook da solo non si
può distinguere un effetto reale dal rumore di ottimizzazione/Monte Carlo
di quella singola run.

**Prima di riportare uno qualunque di questi confronti pre/post, o
qualunque correlazione costruita sopra, come risultato definitivo**, va
rifatto con **almeno 5 seed** -- variando sia il seed di training del
source sia il seed di adattamento -- seguendo la stessa convenzione già
usata per HAR (`SEEDS = [0, 1, 2, 3, 4]`), riportando media ± deviazione
standard sulle run.